# Test GSMSymbBox Expression Simplification

This notebook tests the new expression simplification functionality in GSMSymbBox with timeout handling.

In [9]:
import sympy as sp
import sys
import os
import importlib

# Add the project root to Python path
project_root = '/home/rch/Coding/bmcs_matmod'
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Import and reload modules to get latest changes
import bmcs_matmod.gsm_lagrange.core2.gsm_symb_box as gsm_symb_box_module
import bmcs_matmod.gsm_lagrange.core2.gsm_vars as gsm_vars_module

# Reload the modules to get the latest changes
importlib.reload(gsm_symb_box_module)
importlib.reload(gsm_vars_module)

from bmcs_matmod.gsm_lagrange.core2.gsm_symb_box import GSMSymbBox, StateFunction
from bmcs_matmod.gsm_lagrange.core2.gsm_vars import Scalar

## Test 1: Basic Simplification without Timeout

In [10]:
# Create custom symbols using the Scalar class
T = Scalar('T', codename='T_var')
S = Scalar('S', codename='S_var')
eps = Scalar('epsilon', codename='eps_var')
sig = Scalar('sigma', codename='sig_var')
Eps = Scalar('Epsilon', codename='Eps_var')

print(f"Created symbols: T={T}, S={S}, eps={eps}, sig={sig}, Eps={Eps}")

Created symbols: T=T, S=S, eps=epsilon, sig=sigma, Eps=Epsilon


In [11]:
# Create a simple polynomial expression that can be simplified
# F = T*S + eps*sig + Eps^2 + additional simplifiable terms
simple_expr = T*S + eps*sig + Eps**2 + 2*T + 3*T - 5*T + eps*(sig + 1) - eps

print(f"Original expression: {simple_expr}")

# Create GSMSymbBox with simplification enabled
box_simplified = GSMSymbBox(
    initial_state_fn=StateFunction.HELMHOLTZ,
    T_var=T, S_var=S, 
    eps_vars=(eps), sig_vars=(sig,), 
    Eps_vars=(Eps,),
    initial_expression=simple_expr,
    auto_simplify=True,
    simplify_timeout=2.0  # 2 second timeout
)

print(f"\nSimplified F (Helmholtz): {box_simplified.F}")

Original expression: Epsilon**2 + S*T + epsilon*sigma + epsilon*(sigma + 1) - epsilon

Simplified F (Helmholtz): Epsilon**2 + S*T + 2*epsilon*sigma


## Test 2: Compare with and without Simplification

In [13]:
# Create the same expression without simplification
box_no_simplify = GSMSymbBox(
    initial_state_fn=StateFunction.HELMHOLTZ,
    T_var=T, S_var=S, 
    eps_vars=(eps), sig_vars=(sig,), 
    Eps_vars=(Eps,),
    initial_expression=simple_expr,
    auto_simplify=False  # No simplification
)

print(f"Without simplification: {box_no_simplify.F}")
print(f"With simplification:    {box_simplified.F}")
print(f"\nExpressions are equivalent: {sp.simplify(box_no_simplify.F - box_simplified.F) == 0}")

Without simplification: Epsilon**2 + S*T + epsilon*sigma + epsilon*(sigma + 1) - epsilon
With simplification:    Epsilon**2 + S*T + 2*epsilon*sigma

Expressions are equivalent: True


## Test 3: Legendre Transformation with Simplification

In [14]:
# Test Legendre transformation to other state functions
print("Testing Legendre transformations with simplification:")
print(f"\nHelmholtz F: {box_simplified.F}")

try:
    gibbs = box_simplified.G
    print(f"Gibbs G: {gibbs}")
except Exception as e:
    print(f"Error computing Gibbs: {e}")

try:
    internal_energy = box_simplified.U
    print(f"Internal Energy U: {internal_energy}")
except Exception as e:
    print(f"Error computing Internal Energy: {e}")

try:
    enthalpy = box_simplified.H
    print(f"Enthalpy H: {enthalpy}")
except Exception as e:
    print(f"Error computing Enthalpy: {e}")

Testing Legendre transformations with simplification:

Helmholtz F: Epsilon**2 + S*T + 2*epsilon*sigma
Gibbs G: Epsilon**2 + S*T + epsilon*sigma
Internal Energy U: Epsilon**2 + 2*S*T + 2*epsilon*sigma
Enthalpy H: Epsilon**2 + 2*S*T + 3*epsilon*sigma


## Test 4: Test Timeout Functionality

In [16]:
# Test the timeout mechanism directly
from bmcs_matmod.gsm_lagrange.core2.gsm_symb_box import GSMSymbBox

# Create a box with very short timeout
box_short_timeout = GSMSymbBox(
    initial_state_fn=StateFunction.HELMHOLTZ,
    T_var=T, S_var=S, 
    eps_vars=(eps), sig_vars=(sig,), 
    Eps_vars=(Eps,),
    initial_expression=simple_expr,
    auto_simplify=True,
    simplify_timeout=0.001  # Very short timeout - 1ms
)

print(f"With very short timeout: {box_short_timeout.F}")
print("(Should fall back to original expression if timeout occurs)")

With very short timeout: Epsilon**2 + S*T + 2*epsilon*sigma
(Should fall back to original expression if timeout occurs)


## Test 5: Complex Expression Simplification

In [18]:
# Create a more complex expression that benefits from simplification
complex_expr = (
    T**2 * S + T * S**2 + eps**2 * sig + eps * sig**2 + 
    Eps**3 + (T + 1)**2 - 2*T - 1 + 
    sp.sin(eps)**2 + sp.cos(eps)**2  # This should simplify to 1
)

print(f"Complex original expression: {complex_expr}")

box_complex = GSMSymbBox(
    initial_state_fn=StateFunction.HELMHOLTZ,
    T_var=T, S_var=S, 
    eps_vars=(eps), sig_vars=(sig,), 
    Eps_vars=(Eps,),
    initial_expression=complex_expr,
    auto_simplify=True,
    simplify_timeout=5.0  # Longer timeout for complex expression
)

print(f"\nSimplified complex expression: {box_complex.F}")

Complex original expression: Epsilon**3 + S**2*T + S*T**2 - 2*T + epsilon**2*sigma + epsilon*sigma**2 + (T + 1)**2 + sin(epsilon)**2 + cos(epsilon)**2 - 1

Simplified complex expression: Epsilon**3 + S**2*T + S*T**2 + T**2 + epsilon**2*sigma + epsilon*sigma**2 + 1


## Test 6: Manual Simplification Function Test

In [19]:
# Test the simplify_with_timeout function directly
test_expr = (T + 1)**2 - 2*T - 1 + sp.sin(eps)**2 + sp.cos(eps)**2

print(f"Test expression: {test_expr}")
simplified = box_complex.simplify_with_timeout(test_expr)
print(f"Simplified: {simplified}")

# Verify it's mathematically equivalent
manually_simplified = sp.simplify(test_expr)
print(f"Manual simplify: {manually_simplified}")
print(f"Results match: {sp.simplify(simplified - manually_simplified) == 0}")

Test expression: -2*T + (T + 1)**2 + sin(epsilon)**2 + cos(epsilon)**2 - 1
Simplified: T**2 + 1
Manual simplify: T**2 + 1
Results match: True
